# PySpark Joins

Streaming platform dataset: users, subscriptions, watch events, devices, content catalog, campaign signups, and support tickets.


## 00 — Spark setup and sample data


In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("pyspark-joins-single-notebook")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.warehouse.dir", "/tmp/spark_joins_warehouse")
    .getOrCreate()
)

spark.version


In [ ]:
users = spark.createDataFrame([
    (1, "Anna", "SK", "2024-01-10"),
    (2, "Boris", "CZ", "2024-01-15"),
    (3, "Cyril", "SK", "2024-02-01"),
    (4, "Dana", "AT", "2024-02-20"),
    (5, "Eva", "SK", "2024-03-05"),
    (6, "Filip", "DE", "2024-03-11"),
    (7, "Gina", None, "2024-03-18"),
], ["user_id", "name", "country_code", "registered_at"])

subscriptions = spark.createDataFrame([
    (1, "premium", "active"),
    (2, "basic", "active"),
    (3, "premium", "cancelled"),
    (5, "family", "active"),
    (8, "basic", "active"),
], ["user_id", "plan", "status"])

watch_events = spark.createDataFrame([
    (1001, 1, "c01", "2024-04-01", 35),
    (1002, 1, "c02", "2024-04-01", 20),
    (1003, 2, "c01", "2024-04-02", 10),
    (1004, 2, "c03", "2024-04-02", 55),
    (1005, 3, "c04", "2024-04-03", 12),
    (1006, 5, "c01", "2024-04-04", 40),
    (1007, 5, "c05", "2024-04-05", 18),
    (1008, 8, "c02", "2024-04-05", 22),
    (1009, None, "c06", "2024-04-06", 5),
], ["event_id", "user_id", "content_id", "event_date", "minutes_watched"])

content_catalog = spark.createDataFrame([
    ("c01", "Galaxy Roads", "movie", "sci-fi"),
    ("c02", "Data Detectives", "series", "crime"),
    ("c03", "Cooking Fast", "series", "lifestyle"),
    ("c04", "The Lake", "movie", "drama"),
    ("c05", "Spark Academy", "series", "education"),
], ["content_id", "title", "content_type", "genre"])

devices = spark.createDataFrame([
    (1, "tv", "living-room"),
    (1, "mobile", "anna-phone"),
    (2, "tablet", "boris-ipad"),
    (3, "tv", "bedroom"),
    (5, "mobile", "eva-phone"),
    (5, "tv", "kids-room"),
    (7, "mobile", "gina-phone"),
], ["user_id", "device_type", "device_name"])

campaign_signups = spark.createDataFrame([
    ("spring-2024", 1, "SK"),
    ("spring-2024", 2, "CZ"),
    ("spring-2024", 4, "AT"),
    ("summer-2024", 5, "SK"),
    ("summer-2024", 8, "HU"),
    ("summer-2024", None, "SK"),
], ["campaign_id", "user_id", "country_code"])

support_tickets = spark.createDataFrame([
    (501, 1, "billing", "closed"),
    (502, 2, "playback", "open"),
    (503, 2, "login", "closed"),
    (504, 6, "billing", "open"),
    (505, None, "unknown", "open"),
], ["ticket_id", "user_id", "category", "ticket_status"])

for name, df in {
    "users": users,
    "subscriptions": subscriptions,
    "watch_events": watch_events,
    "content_catalog": content_catalog,
    "devices": devices,
    "campaign_signups": campaign_signups,
    "support_tickets": support_tickets,
}.items():
    df.createOrReplaceTempView(name)
    print(name)
    df.show(truncate=False)


## 01 — Inner join

Users with a subscription. Rows without matching keys on both sides disappear.

```text
users                         subscriptions
+---------+--------+          +---------+----------+
| user_id | name   |          | user_id | plan     |
+---------+--------+          +---------+----------+
|   1     | Anna   |   ───┐   |   1     | premium  |
|   2     | Boris  |   ───┼── |   2     | basic    |
|   3     | Cyril  |   ───┘   |   3     | premium  |
|   4     | Dana   |          |   8     | basic    |
+---------+--------+          +---------+----------+

INNER JOIN on user_id

Result = only matching keys from both sides

+---------+--------+----------+
| user_id | name   | plan     |
+---------+--------+----------+
|   1     | Anna   | premium  |
|   2     | Boris  | basic    |
|   3     | Cyril  | premium  |
+---------+--------+----------+

Dropped:
- users without subscription: user_id 4, 6, 7
- subscriptions without user profile: user_id 8
```


In [ ]:
users.join(subscriptions, on="user_id", how="inner").show(truncate=False)


## 02 — Left, right and full outer joins

Useful for keeping unmatched records from one or both sides.

```text
LEFT JOIN
Keep all rows from users.

users LEFT JOIN subscriptions

users                         subscriptions
+---------+--------+          +---------+----------+
|   1     | Anna   |   ───    |   1     | premium  |
|   2     | Boris  |   ───    |   2     | basic    |
|   4     | Dana   |   ───    |  NULL   | NULL     |
+---------+--------+          +---------+----------+

RIGHT JOIN
Keep all rows from subscriptions.

users RIGHT JOIN subscriptions

users                         subscriptions
+---------+--------+          +---------+----------+
|   1     | Anna   |   ───    |   1     | premium  |
|   2     | Boris  |   ───    |   2     | basic    |
|  NULL   | NULL   |   ───    |   8     | basic    |
+---------+--------+          +---------+----------+

FULL OUTER JOIN
Keep everything from both sides.

+---------+--------+----------+
| user_id | name   | plan     |
+---------+--------+----------+
|   1     | Anna   | premium  |
|   2     | Boris  | basic    |
|   4     | Dana   | NULL     |
|   8     | NULL   | basic    |
+---------+--------+----------+
```


In [ ]:
print("LEFT JOIN: all users, subscription if available")
users.join(subscriptions, on="user_id", how="left").orderBy("user_id").show(truncate=False)

print("RIGHT JOIN: all subscriptions, user profile if available")
users.join(subscriptions, on="user_id", how="right").orderBy("user_id").show(truncate=False)

print("FULL JOIN: everything from both sides")
users.join(subscriptions, on="user_id", how="full").orderBy("user_id").show(truncate=False)


## 03 — Left semi and left anti joins

- `left_semi`: return left rows that have a match on the right
- `left_anti`: return left rows that do not have a match on the right

```text
LEFT SEMI JOIN
Return only columns from the left side.
Keep users that have at least one matching watch event.

users                         watch_events
+---------+--------+          +----------+---------+
| user_id | name   |          | event_id | user_id |
+---------+--------+          +----------+---------+
|   1     | Anna   |   ───    |  1001    |   1     |
|   2     | Boris  |   ───    |  1003    |   2     |
|   4     | Dana   |          |          |         |
+---------+--------+          +----------+---------+

Result:
+---------+--------+
| user_id | name   |
+---------+--------+
|   1     | Anna   |
|   2     | Boris  |
+---------+--------+

LEFT ANTI JOIN
Return only columns from the left side.
Keep users that have no matching watch event.

Result:
+---------+--------+
| user_id | name   |
+---------+--------+
|   4     | Dana   |
|   6     | Filip  |
|   7     | Gina   |
+---------+--------+
```


In [ ]:
print("Users who have watched something")
users.join(watch_events.select("user_id").distinct(), on="user_id", how="left_semi").show(truncate=False)

print("Users with no watch activity")
users.join(watch_events.select("user_id").distinct(), on="user_id", how="left_anti").show(truncate=False)


## 04 — Cross join and self join

Use cross joins carefully. They multiply rows: `left_count * right_count`.

```text
CROSS JOIN
Every plan is combined with every country.

small_plans                  small_countries
+---------+                  +--------------+
| plan    |                  | country_code |
+---------+                  +--------------+
| basic   |  ─────┬───────   | SK           |
| premium |  ─────┼───────   | CZ           |
| family  |  ─────┘          +--------------+
+---------+

3 plans * 2 countries = 6 result rows

SELF JOIN
The same DataFrame is used twice with aliases.

users as left                 users as right
+---------+------+-----+       +---------+------+-----+
| user_id | name | cty |       | user_id | name | cty |
+---------+------+-----+       +---------+------+-----+
|   1     | Anna | SK  | ───   |   3     | Cyril| SK  |
|   1     | Anna | SK  | ───   |   5     | Eva  | SK  |
|   3     | Cyril| SK  | ───   |   5     | Eva  | SK  |
+---------+------+-----+       +---------+------+-----+

Condition:
left.country_code = right.country_code
left.user_id < right.user_id
```


In [ ]:
small_plans = spark.createDataFrame([
    ("basic",),
    ("premium",),
    ("family",),
], ["plan"])

small_countries = spark.createDataFrame([
    ("SK",),
    ("CZ",),
], ["country_code"])

small_plans.crossJoin(small_countries).show(truncate=False)


In [ ]:
print("Self join: users from the same country")
left_users = users.alias("left")
right_users = users.alias("right")

(
    left_users
    .join(
        right_users,
        (F.col("left.country_code") == F.col("right.country_code")) &
        (F.col("left.user_id") < F.col("right.user_id")),
        "inner"
    )
    .select(
        F.col("left.name").alias("user_a"),
        F.col("right.name").alias("user_b"),
        F.col("left.country_code")
    )
    .show(truncate=False)
)


## 05 — Multi-column joins

Join marketing signups to users by both `user_id` and `country_code`.

```text
Single-key join can create false matches when the business identity is composite.

campaign_signups                         users
+-------------+---------+--------------+ +---------+------+--------------+
| campaign_id | user_id | country_code | | user_id | name | country_code |
+-------------+---------+--------------+ +---------+------+--------------+
| spring      |   1     | SK           | |   1     | Anna | SK           |
| spring      |   1     | CZ           | |   1     | Anna | SK           |
+-------------+---------+--------------+ +---------+------+--------------+

Join condition:
campaign_signups.user_id       = users.user_id
campaign_signups.country_code  = users.country_code

Result:
+-------------+---------+--------------+------+
| campaign_id | user_id | country_code | name |
+-------------+---------+--------------+------+
| spring      |   1     | SK           | Anna |
| spring      |   1     | CZ           | NULL |
+-------------+---------+--------------+------+
```


In [ ]:
(
    campaign_signups.alias("c")
    .join(
        users.alias("u"),
        (F.col("c.user_id") == F.col("u.user_id")) &
        (F.col("c.country_code") == F.col("u.country_code")),
        "left"
    )
    .select("c.campaign_id", "c.user_id", "c.country_code", "u.name")
    .orderBy("campaign_id", "user_id")
    .show(truncate=False)
)


## 06 — Nulls and duplicate keys

Normal equality does not match `NULL = NULL`. Use null-safe equality `<=>` in SQL or `eqNullSafe` in the DataFrame API.

```text
NULL HANDLING

Normal equality:
NULL = NULL  -> not matched

watch_events                  support_tickets
+----------+---------+        +-----------+---------+
| event_id | user_id |        | ticket_id | user_id |
+----------+---------+        +-----------+---------+
| 1009     | NULL    |   X    | t03       | NULL    |
+----------+---------+        +-----------+---------+

Null-safe equality:
NULL <=> NULL  -> matched

watch_events                  support_tickets
+----------+---------+        +-----------+---------+
| 1009     | NULL    |  ───   | t03       | NULL    |
+----------+---------+        +-----------+---------+

DUPLICATE KEYS

users                         devices
+---------+------+            +---------+-------------+
| user_id | name |            | user_id | device_type |
+---------+------+            +---------+-------------+
|   1     | Anna |  ───┬───   |   1     | mobile      |
|         |      |     └───   |   1     | tv          |
+---------+------+            +---------+-------------+

One left row * two right matches = two result rows.
```


In [ ]:
print("Normal join: NULL keys do not match")
(
    watch_events.alias("w")
    .join(support_tickets.alias("t"), F.col("w.user_id") == F.col("t.user_id"), "inner")
    .select("w.event_id", "w.user_id", "t.ticket_id", "t.category")
    .show(truncate=False)
)

print("Null-safe join: NULL keys can match")
(
    watch_events.alias("w")
    .join(support_tickets.alias("t"), F.col("w.user_id").eqNullSafe(F.col("t.user_id")), "inner")
    .select("w.event_id", "w.user_id", "t.ticket_id", "t.category")
    .show(truncate=False)
)


In [ ]:
print("Duplicate keys multiply rows")
(
    users
    .join(devices, on="user_id", how="inner")
    .select("user_id", "name", "device_type", "device_name")
    .orderBy("user_id", "device_type")
    .show(truncate=False)
)


## 07 — Broadcast join

Broadcast small dimension tables such as `content_catalog` to avoid shuffling the large fact table.

```text
Without broadcast
Both sides may be shuffled by join key.

watch_events partitions            content_catalog partitions
+-----------+                       +--------------+
| partition |  shuffle by content_id| partition    |
+-----------+  ───────────────────  +--------------+
| p1        |                       | p1           |
| p2        |                       | p2           |
| p3        |                       | p3           |
+-----------+                       +--------------+

With broadcast
Small dimension table is copied to each executor.

                  broadcast(content_catalog)
                         +--------------+
                         | small table  |
                         +--------------+
                          /      |                               /       |       +----------------+   +----------------+   +----------------+
| executor 1     |   | executor 2     |   | executor 3     |
| watch_events   |   | watch_events   |   | watch_events   |
| local join     |   | local join     |   | local join     |
+----------------+   +----------------+   +----------------+

Expected physical plan signal:
BroadcastHashJoin
```


In [ ]:
joined = (
    watch_events
    .join(F.broadcast(content_catalog), on="content_id", how="left")
    .select("event_id", "user_id", "title", "genre", "minutes_watched")
)

joined.show(truncate=False)
joined.explain(mode="formatted")


## 08 — Join strategies and explain plan

Compare physical plans with and without a broadcast hint.

```text
Logical join request

watch_events JOIN content_catalog ON content_id

Spark planner chooses a physical strategy:

+----------------------+---------------------------------------------+
| Strategy             | Typical use                                  |
+----------------------+---------------------------------------------+
| BroadcastHashJoin    | One side is small enough to broadcast        |
| SortMergeJoin        | Large tables, sortable join keys             |
| ShuffledHashJoin     | Hash join after shuffle, selected by planner |
| BroadcastNestedLoop  | Non-equi joins or cross joins                |
+----------------------+---------------------------------------------+

explain(mode="formatted") shows:

Parsed Logical Plan
        ↓
Analyzed Logical Plan
        ↓
Optimized Logical Plan
        ↓
Physical Plan
        ↓
Selected join operator
```


In [ ]:
print("Without explicit broadcast hint")
watch_events.join(content_catalog, on="content_id", how="inner").explain(mode="formatted")

print("With explicit broadcast hint")
watch_events.join(F.broadcast(content_catalog), on="content_id", how="inner").explain(mode="formatted")


## 09 — Skewed join problem

This small example simulates one very frequent key. In real data, skew can cause one task to process much more data than others.

```text
SKEWED KEY DISTRIBUTION

skewed_events
+---------+-------+
| user_id | rows  |
+---------+-------+
|   1     | 100   |  ← hot key
|   2     |   1   |
|   3     |   1   |
|   5     |   1   |
+---------+-------+

Normal join by user_id

Partition after shuffle:
+-------------+------------------------------+
| partition 1 | user_id 1: 100 rows          |  ← heavy task
| partition 2 | user_id 2:   1 row           |
| partition 3 | user_id 3:   1 row           |
| partition 4 | user_id 5:   1 row           |
+-------------+------------------------------+

SALTING PATTERN

Large side: split hot key across salt buckets.

skewed_events
+---------+------+       +---------+------+
| user_id | rows |  ->   | user_id | salt |
+---------+------+       +---------+------+
|   1     | 100  |       |   1     | 0..3 |
+---------+------+       +---------+------+

Small side: duplicate hot key for all salt buckets.

users
+---------+------+       +---------+------+
| user_id | name |  ->   | user_id | salt |
+---------+------+       +---------+------+
|   1     | Anna |       |   1     | 0    |
|         |      |       |   1     | 1    |
|         |      |       |   1     | 2    |
|         |      |       |   1     | 3    |
+---------+------+       +---------+------+

Join condition becomes:
user_id + salt
```


In [ ]:
skewed_events = spark.createDataFrame(
    [(i, 1, "c01", 1) for i in range(1, 101)] +
    [(101, 2, "c02", 1), (102, 3, "c03", 1), (103, 5, "c04", 1)],
    ["event_id", "user_id", "content_id", "minutes_watched"]
)

print("Key distribution")
skewed_events.groupBy("user_id").count().orderBy(F.desc("count")).show()

print("Join result still correct, but key 1 is much heavier")
skewed_events.join(users, on="user_id", how="left").groupBy("user_id", "name").count().orderBy(F.desc("count")).show()


In [ ]:
print("Simple salting pattern for a skewed key")
SALT_BUCKETS = 4

salted_events = skewed_events.withColumn(
    "salt",
    F.when(F.col("user_id") == 1, (F.rand(seed=42) * SALT_BUCKETS).cast("int")).otherwise(F.lit(0))
)

salt_array = F.when(
    F.col("user_id") == 1,
    F.array(*[F.lit(i) for i in range(SALT_BUCKETS)])
).otherwise(F.array(F.lit(0)))

salted_users = users.withColumn("salt", F.explode(salt_array))

(
    salted_events
    .join(salted_users, on=["user_id", "salt"], how="left")
    .groupBy("user_id", "name")
    .count()
    .orderBy(F.desc("count"))
    .show()
)


## 10 — Bucketed joins optional

Bucketed joins can reduce shuffle when both sides are saved bucketed by the join key and reused across workloads. This is optional and depends on table format, catalog, and Spark configuration.

```text
BUCKETED TABLES

Both tables are written with the same bucket count and same bucket column.

bucketBy(4, "user_id")

bucketed_watch_events              bucketed_users
+---------------------+            +----------------+
| bucket 0: user_id %4|    join    | bucket 0       |
| bucket 1: user_id %4|  ───────   | bucket 1       |
| bucket 2: user_id %4|            | bucket 2       |
| bucket 3: user_id %4|            | bucket 3       |
+---------------------+            +----------------+

Potential benefit:
matching keys are already grouped into compatible buckets.

Important:
- useful when bucketed tables are reused
- not always faster for small data
- catalog/table format/configuration can change the physical plan
- verify with explain(mode="formatted")
```


In [ ]:
import shutil
from pathlib import Path

bucket_db = "joins_demo"
bucket_db_path = Path("/tmp/spark_joins_warehouse") / f"{bucket_db}.db"

spark.sql("DROP TABLE IF EXISTS joins_demo.bucketed_watch_events")
spark.sql("DROP TABLE IF EXISTS joins_demo.bucketed_users")
spark.sql("DROP DATABASE IF EXISTS joins_demo")
shutil.rmtree(bucket_db_path, ignore_errors=True)

spark.sql(f"CREATE DATABASE joins_demo LOCATION '{bucket_db_path.as_posix()}'")
spark.catalog.setCurrentDatabase(bucket_db)
spark.conf.set("spark.sql.sources.bucketing.enabled", "true")

(
    watch_events
    .write
    .mode("overwrite")
    .bucketBy(4, "user_id")
    .sortBy("user_id")
    .saveAsTable("bucketed_watch_events")
)

(
    users
    .write
    .mode("overwrite")
    .bucketBy(4, "user_id")
    .sortBy("user_id")
    .saveAsTable("bucketed_users")
)

bucketed_join = spark.table("joins_demo.bucketed_watch_events").join(
    spark.table("joins_demo.bucketed_users"),
    on="user_id",
    how="inner"
)

bucketed_join.show(truncate=False)
bucketed_join.explain(mode="formatted")


## Cleanup


In [ ]:
spark.sql("DROP TABLE IF EXISTS joins_demo.bucketed_watch_events")
spark.sql("DROP TABLE IF EXISTS joins_demo.bucketed_users")
spark.sql("DROP DATABASE IF EXISTS joins_demo")
